In [ ]:
import os
import torch
import gc
from dotenv import load_dotenv
from transformers import pipeline, BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM, TextStreamer
from huggingface_hub import login
import gradio as gr

In [ ]:
# open-source models

LLAMA_3_1_8B = "meta-llama/Llama-3.1-8B-Instruct"
LLAMA_3_2_3B = "meta-llama/Llama-3.2-3B-Instruct"
LLAMA_3_2_1B = "meta-llama/Llama-3.2-1B-Instruct"
GEMMA = "google/gemma-3-270m-it"
PHI = "microsoft/Phi-4-mini-instruct"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [ ]:
load_dotenv()

hf_token = os.getenv("HF_TOKEN")

In [ ]:
messages = [
    {"role": "user", "content": "Create a dataset of 5 objects each being unique. Each object should have a first name, last name, and email. Provide in json format."}
]

In [ ]:
# quantization will reduces memory usage at the cost of precision

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
# tokenize the inputs

tokenizer = AutoTokenizer.from_pretrained(LLAMA_3_2_3B)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

In [ ]:
inputs

In [ ]:
model = AutoModelForCausalLM.from_pretrained(LLAMA_3_2_3B, device_map="auto", quantization_config=quant_config)

In [ ]:
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")

In [ ]:
model

In [ ]:
outputs = model.generate(**inputs, max_new_tokens=80)
outputs[0]

In [ ]:
tokenizer.decode(outputs[0])

In [ ]:
del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()

In [ ]:
def generate(model, messages, quant=True, max_new_tokens=80):
    tokenizer = AutoTokenizer.from_pretrained(model)
    tokenizer.pad_token = tokenizer.eos_token
    input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True, return_dict=False).to("cuda")
    attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")
    streamer = TextStreamer(tokenizer)
    
    if quant:
        model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config).to("cuda")
    else:
        model = AutoModelForCausalLM.from_pretrained(model).to("cuda")
    outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)
    

In [ ]:
generate(GEMMA, messages, False, 600)

In [ ]:
generator = pipeline("text-generation", device="cuda")
result = generator(messages)
print(result)